> **Jupyter 학습 안내**
>
> 이 Notebook은 Course의 상세 학습노트입니다. 본문과 수식을 먼저 읽고 Python 예제 셀을 한 단계씩 실행하세요.
> 코드 셀은 개념을 보여주는 작은 예제로, 필요한 데이터와 변수는 바로 앞 설명을 확인해야 합니다.
> 처음부터 끝까지 실행하는 통합 실습은 저장소의 notebooks와 notebooks/data_analysis를 사용합니다.
> 예제 결과를 예상한 뒤 실행하고, 값·조건·열 이름을 바꾸어 결과 차이를 기록하세요.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "courses").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 Notebook을 실행하세요.")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("학습 저장소:", ROOT)

# Course 10 학습노트: 서비스 통합, 검증과 발표

## 1. 학습목표

최종 단계의 목표는 화면을 더 추가하는 것이 아니라 데이터 수집부터 사용자 결과까지 전체 경로가 정확하고 재현 가능하며 설명 가능한지 검증하는 것이다. 권장 시간은 통합 180분, 테스트 120분, 사용자평가 90분, 발표 준비 120분이다.

## 2. 전체 파이프라인

```text
공식 출처 → 원본 저장 → 품질 진단 → 정제·통합 → EDA
         → 특성·모델 → 검증 → DB → Streamlit → 사용자 피드백
```

각 화살표에 입력, 출력, 책임 함수, 검증 조건이 있어야 한다. 수작업으로 파일을 고친 단계가 있으면 재현할 수 없다.

## 3. Definition of Done

기능 하나의 완료 기준 예시는 다음과 같다.

- 요구사항과 정상 입력이 정의되어 있다.
- 예상 가능한 오류와 빈 결과를 처리한다.
- 자동 테스트 또는 재현 가능한 수동 검증이 있다.
- 화면에서 출처, 기준일, 단위를 확인할 수 있다.
- 다른 팀원이 README만 보고 실행했다.
- 관련 코드와 문서가 Pull Request에서 검토되었다.

## 4. 테스트 전략

### 단위 테스트

정규화, 가중치 합, 전처리처럼 작은 함수의 입출력을 확인한다.

In [ ]:
def test_normalized_score_range():
    result = calculate_scores(sample)
    assert result["적합도"].between(0, 100).all()

### 데이터 테스트

- 식별자 유일성
- 필수 열 존재
- 허용 범위
- 병합 성공률
- 기준일 일관성

### 통합 테스트

샘플 원본에서 최종 추천 CSV까지 한 번에 실행하고 행 수·열·상위 순위를 확인한다.

### 사용자 테스트

사용자에게 설명하지 않고 “카페 후보지 3곳과 근거를 찾으세요” 같은 과업을 준다. 성공 여부, 시간, 망설임, 오류와 발화를 기록한다.

## 5. 모델·데이터 검증

- 가중치 시나리오별 순위 안정성
- 이상값 포함·제외 결과 비교
- 결측 대체 방법별 결과 비교
- 데이터 기준월 변화에 따른 결과 비교
- 누락 변수와 적용 불가능한 상황 기록

추천 결과에 실제 정답이 없다면 내부 정확도처럼 보이는 숫자를 만들지 않는다. 설명가능성, 안정성, 재현성, 사용자 유용성을 평가한다.

## 6. 윤리와 책임

상권 추천은 자본·정책 결정에 영향을 줄 수 있다. 다음을 화면과 보고서에 포함한다.

- 합성 데이터와 실제 데이터 구분
- 데이터 출처·라이선스·기준일
- 추천에 포함되지 않은 중요한 변수
- 특정 지역에 불리한 지표·가중치
- 개인정보와 정밀 위치 노출 위험
- 실제 결정 전 전문가 검토 필요성

## 7. 성능과 운영

- 큰 원본을 매 실행마다 읽지 않고 전처리 결과를 사용한다.
- 데이터 로딩에는 캐시를 사용한다.
- API는 화면 요청마다 호출하지 않고 갱신 작업으로 분리한다.
- 로그에는 실행시각, 데이터 버전, 오류를 남기되 키와 개인정보는 제외한다.
- `requirements.txt` 버전과 실행 명령을 유지한다.

## 8. README 작성

처음 방문한 사람이 다음 순서로 이해해야 한다.

1. 문제와 대상 사용자
2. 주요 화면 또는 데모
3. 데이터 출처와 제한
4. 설치·실행 명령
5. 폴더 구조
6. 분석·추천 방법
7. 테스트 방법
8. 팀원별 기여

## 9. 최종 발표 구조

| 시간 | 내용 |
|---:|---|
| 1분 | 문제와 사용자 |
| 2분 | 데이터 품질과 핵심 발견 |
| 2분 | 추천 방법과 검증 |
| 3분 | 실제 서비스 시연 |
| 1분 | 한계·윤리·실패 조건 |
| 1분 | 결론과 개선 계획 |

시연은 성공 경로만 녹화하지 말고 입력을 바꾸어 결과가 실제로 갱신되는 것을 보여준다.

## 10. 개인 역량 확인

팀 프로젝트와 별도로 Lab 07을 수행한다. 교수자는 제출 노트북의 임의 셀을 선택하여 설명하게 하고 변수·조건·임계값을 바꾸어 5분 안에 재실행하도록 요청한다.

## 11. 최종 체크리스트

- [ ] 새 환경에서 설치·실행했다.
- [ ] 모든 Notebook을 Restart & Run All 했다.
- [ ] 자동 테스트와 데이터 테스트를 통과했다.
- [ ] API 키·개인정보·대용량 원본이 커밋되지 않았다.
- [ ] 추천 근거, 가중치, 기준일, 한계가 화면에 있다.
- [ ] 사용자 테스트 결과를 반영했다.
- [ ] 팀원별 커밋·리뷰·개인 실기 증거가 있다.
- [ ] 실제 적용 전 필요한 추가 검증을 발표한다.

## 학습 마무리

1. 이 Course의 핵심 개념 세 가지를 본인의 말로 정리한다.
2. 수식 하나를 작은 숫자로 손계산하고 Python 결과와 비교한다.
3. 예제 코드의 입력이나 조건을 하나 바꾸어 결과 차이를 설명한다.
4. quiz.md에 먼저 답한 뒤 quiz_answer.md와 비교한다.
5. assignment.md와 연결된 실습 Notebook을 Restart & Run All로 확인한다.